In [1]:
#Import

In [2]:
import pickle
import os
from dataclasses import dataclass, field
from datetime import datetime
from typing import List, Dict, Optional

import numpy as np

DATA_FILE = "bank_data.pkl" 

In [ ]:
#Custom Exceptions

In [5]:
class BankError(Exception):
    """Base class for all bank-system errors."""
    pass


class AccountNotFoundError(BankError):
    """Used when an account is not found"""
    pass


class InvalidAmountError(BankError):
    """used when amount is not valid"""
    pass


class InsufficientFundsError(BankError):
    """used when amount is not enough balance"""
    pass


class AuthenticationError(BankError):
    """Used when the password is incorrect"""
    pass

In [ ]:
#Data Model

In [6]:
@dataclass
class Transaction:
    """Stores details of one transaction"""
    timestamp: str
    ttype: str            # deposit | withdrawal | transfer_out | transfer_in
    amount: float
    balance_after: float
    counterparty: Optional[str] = None   # other account number for transfers

    def to_row(self):
        """Convert the trasaction details into a tuple"""
        return (self.timestamp, self.ttype, self.amount, self.balance_after, self.counterparty)


@dataclass
class Account:
    """Stores account details and transaction history"""
    account_number: str
    holder_name: str
    account_type: str                 # savings | current
    balance: float
    username: Optional[str] = None
    password: Optional[str] = None
    transactions: List[Transaction] = field(default_factory=list)

    def add_transaction(self, ttype: str, amount: float, counterparty: Optional[str] = None):
        self.transactions.append(
            Transaction(
                timestamp=datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                ttype=ttype,
                amount=amount,
                balance_after=self.balance,
                counterparty=counterparty,
            )
        )

In [ ]:
#File handling

In [8]:
def load_accounts(filepath: str = DATA_FILE) -> Dict[str, Account]:
    """Load the accounts dict from a pickle file and returns {} if no file exists yet"""
    if not os.path.exists(filepath):
        return {}
    try:
        with open(filepath, "rb") as f:
            data = pickle.load(f)
            return data if isinstance(data, dict) else {}
    except (pickle.UnpicklingError, EOFError):
        # Corrupt or empty file: fail safe with an empty store rather than crashing.
        print(f"Warning: {filepath!r} could not be read; starting with an empty store.")
        return {}


def save_accounts(accounts: Dict[str, Account], filepath: str = DATA_FILE) -> None:
    """save accounts to the pickle file"""
    tmp_path = filepath + ".tmp"
    with open(tmp_path, "wb") as f:
        pickle.dump(accounts, f)
    os.replace(tmp_path, filepath)   

In [ ]:
#Account management

In [9]:
def generate_account_number(accounts: Dict[str, Account]) -> str:
    """Auto-generate the next account number..."""
    if not accounts:
        return "ACC1001"
    max_num = max(int(acc_no.replace("ACC", "")) for acc_no in accounts.keys())
    return f"ACC{max_num + 1}"


def create_account(accounts: Dict[str, Account], holder_name: str, account_type: str,
                    initial_balance: float, username: Optional[str] = None,
                    password: Optional[str] = None) -> str:
    """Create a new account and returns its account number"""

    if not holder_name or not holder_name.strip():
        raise ValueError("Account holder name cannot be empty")
    if account_type.lower() not in ("savings", "current"):
        raise ValueError("Account type must be savings or current")
    if initial_balance < 0:
        raise InvalidAmountError("Initial balance cannot be negative")

    acc_no = generate_account_number(accounts)
    account = Account(
        account_number=acc_no,
        holder_name=holder_name.strip(),
        account_type=account_type.lower(),
        balance=round(float(initial_balance), 2),
        username=username,
        password=password,
    )
    if initial_balance > 0:
        account.add_transaction("deposit", round(float(initial_balance), 2))
    accounts[acc_no] = account
    return acc_no


def get_account(accounts: Dict[str, Account], account_number: str) -> Account:
    """Find an account using account number"""
    if account_number not in accounts:
        raise AccountNotFoundError(f"No account found with number {account_number!r}.")
    return accounts[account_number]


def get_account_details(accounts: Dict[str, Account], account_number: str) -> dict:
    """Return the main details of an account"""
    acc = get_account(accounts, account_number)
    return {
        "account_number": acc.account_number,
        "holder_name": acc.holder_name,
        "account_type": acc.account_type,
        "balance": round(acc.balance, 2),
        "num_transactions": len(acc.transactions),
    }


def print_account_details(accounts: Dict[str, Account], account_number: str) -> None:
    d = get_account_details(accounts, account_number)
    print("-" * 42)
    print(" Account Number : " + str(d["account_number"]))
    print(" Holder Name    : " + str(d["holder_name"]))
    print(" Account Type   : " + str(d["account_type"]).capitalize())
    print(f" Balance        : Rs. {d['balance']:,.2f}")
    print(" Transactions   : " + str(d["num_transactions"]))
    print("-" * 42)

In [ ]:
#Trasactions

In [10]:
def _validate_amount(amount: float) -> float:
    """Check if the amount is valid"""
    try:
        amount = float(amount)
    except (TypeError, ValueError):
        raise InvalidAmountError("Amount must be a number")
    if amount != amount or amount in (float("inf"), float("-inf")):   # ccheck for Nan or infinity
        raise InvalidAmountError("Amount must be a finite number")
    if amount <= 0:
        raise InvalidAmountError("Amount must be strictly greater than zero")
    return round(amount, 2)


def deposit(accounts: Dict[str, Account], account_number: str, amount: float) -> float:
    """Deposit money and returns the new balance"""
    amount = _validate_amount(amount)
    acc = get_account(accounts, account_number)
    acc.balance = round(acc.balance + amount, 2)
    acc.add_transaction("deposit", amount)
    return acc.balance


def withdraw(accounts: Dict[str, Account], account_number: str, amount: float) -> float:
    """Withdraw money and returns the new balance"""
    amount = _validate_amount(amount)
    acc = get_account(accounts, account_number)
    if amount > acc.balance:
        raise InsufficientFundsError(
            f"Withdrawal of Rs. {amount:,.2f} exceeds available balance of Rs. {acc.balance:,.2f}."
        )
    acc.balance = round(acc.balance - amount, 2)
    acc.add_transaction("withdrawal", amount)
    return acc.balance


def transfer(accounts: Dict[str, Account], from_account: str, to_account: str, amount: float) -> tuple:
    """Transfer money between two accounts and return their balances"""
    amount = _validate_amount(amount)
    if from_account == to_account:
        raise ValueError("Cannot transfer to the same account")

    src = get_account(accounts, from_account)   # raises AccountNotFoundError if missing
    dst = get_account(accounts, to_account)      # raises AccountNotFoundError if missing

    if amount > src.balance:
        raise InsufficientFundsError(
            f"Transfer of Rs. {amount:,.2f} exceeds available balance of Rs. {src.balance:,.2f}."
        )

    # Mutate both sides only after both accounts are confirmed valid and funds are sufficient
    src.balance = round(src.balance - amount, 2)
    dst.balance = round(dst.balance + amount, 2)
    src.add_transaction("transfer_out", amount, counterparty=to_account)
    dst.add_transaction("transfer_in", amount, counterparty=from_account)
    return src.balance, dst.balance

In [ ]:
#Reports and statistics

In [11]:
def transaction_history(accounts: Dict[str, Account], account_number: str) -> None:
    """Show a transaction history of an account"""
    acc = get_account(accounts, account_number)
    if not acc.transactions:
        print(f"No transactions yet for {account_number}.")
        return

    print(f"\nTransaction History -- {account_number} ({acc.holder_name})")
    header = "{:<20}{:<15}{:>12}{:>16}{:>15}".format("Date", "Type", "Amount", "Balance After", "Counterparty")
    print(header)
    print("-" * 78)
    for t in acc.transactions:
        cp = t.counterparty or "-"
        row = "{:<20}{:<15}{:>12,.2f}{:>16,.2f}{:>15}".format(t.timestamp, t.ttype, t.amount, t.balance_after, cp)
        print(row)


def account_summary_stats(accounts: Dict[str, Account], account_number: str) -> dict:
    """Calculate summary statistics using NumPy"""
    acc = get_account(accounts, account_number)

    
    is_deposit = lambda t: t.ttype == "deposit"
    is_withdrawal = lambda t: t.ttype in ("withdrawal", "transfer_out")
    is_inflow = lambda t: t.ttype in ("deposit", "transfer_in")

    deposits = np.array([t.amount for t in acc.transactions if is_deposit(t)], dtype=float)
    withdrawals = np.array([t.amount for t in acc.transactions if is_withdrawal(t)], dtype=float)
    all_amounts = np.array([t.amount for t in acc.transactions if is_inflow(t) or is_withdrawal(t)], dtype=float)

    stats = {
        "total_deposits": float(np.sum(deposits)) if deposits.size else 0.0,
        "total_withdrawals": float(np.sum(withdrawals)) if withdrawals.size else 0.0,
        "avg_transaction_amount": float(np.mean(all_amounts)) if all_amounts.size else 0.0,
        "max_transaction_amount": float(np.max(all_amounts)) if all_amounts.size else 0.0,
        "std_transaction_amount": float(np.std(all_amounts)) if all_amounts.size else 0.0,
        "transaction_count": int(all_amounts.size),
    }
    return stats


def print_summary_stats(accounts: Dict[str, Account], account_number: str) -> None:
    s = account_summary_stats(accounts, account_number)
    print(f"\nSummary Statistics -- {account_number}")
    print("-" * 42)
    print(f" Total Deposits          : Rs. {s['total_deposits']:,.2f}")
    print(f" Total Withdrawals       : Rs. {s['total_withdrawals']:,.2f}")
    print(f" Average Txn Amount      : Rs. {s['avg_transaction_amount']:,.2f}")
    print(f" Largest Txn Amount      : Rs. {s['max_transaction_amount']:,.2f}")
    print(f" Txn Amount Std. Dev.    : Rs. {s['std_transaction_amount']:,.2f}")
    print(" Total Transaction Count : " + str(s["transaction_count"]))
    print("-" * 42)

In [ ]:
#Login support

In [12]:
def authenticate(accounts: Dict[str, Account], account_number: str, password: str) -> bool:
    """Return True if the password is correct"""
    
    acc = get_account(accounts, account_number)

    if acc.password is None:
        raise AuthenticationError(f"Account {account_number} has no password configured.")
    if acc.password != password:
        raise AuthenticationError("Incorrect password.")
    return True

In [ ]:
#Main Menu 

In [13]:
def _prompt_float(msg: str) -> float:
    while True:
        raw = input(msg).strip()
        try:
            return float(raw)
        except ValueError:
            print("Please enter a valid number.")


def main_menu(accounts: Optional[Dict[str, Account]] = None, filepath: str = DATA_FILE) -> Dict[str, Account]:
    """Run the main menu and returns the final accounts dict"""
    if accounts is None:
        accounts = load_accounts(filepath)

    MENU = """
============================================
   BANK ACCOUNT MANAGEMENT SYSTEM
============================================
 1. Open a new account
 2. View account details
 3. Deposit
 4. Withdraw
 5. Transfer funds
 6. View transaction history
 7. View summary statistics
 8. Exit
============================================"""

    while True:
        print(MENU)
        choice = input("Select an option (1-8): ").strip()

        try:
            if choice == "1":
                name = input("Account holder's name: ")
                atype = input("Account type (savings/current): ")
                bal = _prompt_float("Initial balance: ")
                acc_no = create_account(accounts, name, atype, bal)
                print(f"Account created successfully. Account number: {acc_no}")

            elif choice == "2":
                acc_no = input("Account number: ").strip()
                print_account_details(accounts, acc_no)

            elif choice == "3":
                acc_no = input("Account number: ").strip()
                amt = _prompt_float("Deposit amount: ")
                new_bal = deposit(accounts, acc_no, amt)
                print(f"Deposit successful. New balance: Rs. {new_bal:,.2f}")

            elif choice == "4":
                acc_no = input("Account number: ").strip()
                amt = _prompt_float("Withdrawal amount: ")
                new_bal = withdraw(accounts, acc_no, amt)
                print(f"Withdrawal successful. New balance: Rs. {new_bal:,.2f}")

            elif choice == "5":
                src = input("From account number: ").strip()
                dst = input("To account number: ").strip()
                amt = _prompt_float("Transfer amount: ")
                b1, b2 = transfer(accounts, src, dst, amt)
                print(f"Transfer successful. {src} balance: Rs. {b1:,.2f} | {dst} balance: Rs. {b2:,.2f}")

            elif choice == "6":
                acc_no = input("Account number: ").strip()
                transaction_history(accounts, acc_no)

            elif choice == "7":
                acc_no = input("Account number: ").strip()
                print_summary_stats(accounts, acc_no)

            elif choice == "8":
                save_accounts(accounts, filepath)
                print("Data saved. Goodbye!")
                break

            else:
                print("Invalid option. Please choose 1-8.")

        except BankError as e:
            print(f"Error: {e}")
        except ValueError as e:
            print(f"Invalid input: {e}")

        # persist after every mutating action so no work is lost on crash/interrupt
        save_accounts(accounts, filepath)

    return accounts

In [ ]:
#sample run

In [15]:

if os.path.exists(DATA_FILE):
    os.remove(DATA_FILE)

accounts = load_accounts()
print(f"Loaded {len(accounts)} account(s) from {DATA_FILE!r}.")

Loaded 0 account(s) from 'bank_data.pkl'.


In [ ]:
#opening 2 new accounts demo

In [16]:
acc1 = create_account(accounts, "Bhanwar Singh", "savings", 50000)
acc2 = create_account(accounts, "Kavita Kanwar", "current", 200000)

print(f"Created account {acc1} for Bhanwar Singh (savings)")
print(f"Created account {acc2} for Kavita Kanwar (current)")

print_account_details(accounts, acc1)
print_account_details(accounts, acc2)

Created account ACC1001 for Bhanwar Singh (savings)
Created account ACC1002 for Kavita Kanwar (current)
------------------------------------------
 Account Number : ACC1001
 Holder Name    : Bhanwar Singh
 Account Type   : Savings
 Balance        : Rs. 50,000.00
 Transactions   : 1
------------------------------------------
------------------------------------------
 Account Number : ACC1002
 Holder Name    : Kavita Kanwar
 Account Type   : Current
 Balance        : Rs. 200,000.00
 Transactions   : 1
------------------------------------------


In [ ]:
#deposit & withdrawals demo

In [17]:
deposit(accounts, acc1, 1500)
withdraw(accounts, acc1, 800)
deposit(accounts, acc2, 750)

print_account_details(accounts, acc1)
print_account_details(accounts, acc2)

------------------------------------------
 Account Number : ACC1001
 Holder Name    : Bhanwar Singh
 Account Type   : Savings
 Balance        : Rs. 50,700.00
 Transactions   : 3
------------------------------------------
------------------------------------------
 Account Number : ACC1002
 Holder Name    : Kavita Kanwar
 Account Type   : Current
 Balance        : Rs. 200,750.00
 Transactions   : 2
------------------------------------------


In [ ]:
#transfer demo

In [18]:
b1, b2 = transfer(accounts, acc1, acc2, 1200)
print(f"After transfer -> {acc1}: Rs. {b1:,.2f}   {acc2}: Rs. {b2:,.2f}")

After transfer -> ACC1001: Rs. 49,500.00   ACC1002: Rs. 201,950.00


In [ ]:
#Error handling demo

In [19]:
for label, fn in [
    ("Withdraw more than balance", lambda: withdraw(accounts, acc1, 999999)),
    ("Deposit a negative amount",  lambda: deposit(accounts, acc1, -50)),
    ("Transfer to unknown account", lambda: transfer(accounts, acc1, "ACC9999", 100)),
]:
    try:
        fn()
    except BankError as e:
        print(f"[{label}] Caught expected error -> {type(e).__name__}: {e}")

[Withdraw more than balance] Caught expected error -> InsufficientFundsError: Withdrawal of Rs. 999,999.00 exceeds available balance of Rs. 49,500.00.
[Deposit a negative amount] Caught expected error -> InvalidAmountError: Amount must be strictly greater than zero
[Transfer to unknown account] Caught expected error -> AccountNotFoundError: No account found with number 'ACC9999'.


In [ ]:
#transaction history demo

In [20]:
transaction_history(accounts, acc1)
transaction_history(accounts, acc2)


Transaction History -- ACC1001 (Bhanwar Singh)
Date                Type                 Amount   Balance After   Counterparty
------------------------------------------------------------------------------
2026-10-05 19:40:31 deposit           50,000.00       50,000.00              -
2026-10-05 19:41:41 deposit            1,500.00       51,500.00              -
2026-10-05 19:41:41 withdrawal           800.00       50,700.00              -
2026-10-05 19:42:29 transfer_out       1,200.00       49,500.00        ACC1002

Transaction History -- ACC1002 (Kavita Kanwar)
Date                Type                 Amount   Balance After   Counterparty
------------------------------------------------------------------------------
2026-10-05 19:40:31 deposit          200,000.00      200,000.00              -
2026-10-05 19:41:41 deposit              750.00      200,750.00              -
2026-10-05 19:42:29 transfer_in        1,200.00      201,950.00        ACC1001


In [ ]:
#Numpy summary statistics 

In [21]:
print_summary_stats(accounts, acc1)
print_summary_stats(accounts, acc2)


Summary Statistics -- ACC1001
------------------------------------------
 Total Deposits          : Rs. 51,500.00
 Total Withdrawals       : Rs. 2,000.00
 Average Txn Amount      : Rs. 13,375.00
 Largest Txn Amount      : Rs. 50,000.00
 Txn Amount Std. Dev.    : Rs. 21,146.91
 Total Transaction Count : 4
------------------------------------------

Summary Statistics -- ACC1002
------------------------------------------
 Total Deposits          : Rs. 200,750.00
 Total Withdrawals       : Rs. 0.00
 Average Txn Amount      : Rs. 67,316.67
 Largest Txn Amount      : Rs. 200,000.00
 Txn Amount Std. Dev.    : Rs. 93,821.46
 Total Transaction Count : 3
------------------------------------------


In [ ]:
#Persistent Storage : Restart and Reload

In [22]:
save_accounts(accounts)
print(f"Saved {len(accounts)} account(s) to {DATA_FILE!r}.")

del accounts
reloaded = load_accounts()
print(f"Reloaded {len(reloaded)} account(s) from disk after simulated restart.")
print_account_details(reloaded, acc1)

Saved 2 account(s) to 'bank_data.pkl'.
Reloaded 2 account(s) from disk after simulated restart.
------------------------------------------
 Account Number : ACC1001
 Holder Name    : Bhanwar Singh
 Account Type   : Savings
 Balance        : Rs. 49,500.00
 Transactions   : 4
------------------------------------------


In [ ]:
#Simulated Menu Session

In [23]:
import builtins

scripted_inputs = iter([
    "1", "Aarav Mehta", "savings", "10000",     # open a new account
    "6", "ACC1003",                              # view its transaction history
    "3", "ACC1003", "2500",                      # deposit
    "7", "ACC1003",                               # summary stats
    "8",                                          # exit
])

original_input = builtins.input
builtins.input = lambda prompt="": next(scripted_inputs)

try:
    menu_accounts = main_menu(reloaded, filepath=DATA_FILE)
finally:
    builtins.input = original_input   # always restore the real input()

print("\nMenu session complete. Final account count:", len(menu_accounts))


   BANK ACCOUNT MANAGEMENT SYSTEM
 1. Open a new account
 2. View account details
 3. Deposit
 4. Withdraw
 5. Transfer funds
 6. View transaction history
 7. View summary statistics
 8. Exit
Account created successfully. Account number: ACC1003

   BANK ACCOUNT MANAGEMENT SYSTEM
 1. Open a new account
 2. View account details
 3. Deposit
 4. Withdraw
 5. Transfer funds
 6. View transaction history
 7. View summary statistics
 8. Exit

Transaction History -- ACC1003 (Aarav Mehta)
Date                Type                 Amount   Balance After   Counterparty
------------------------------------------------------------------------------
2026-10-05 19:50:49 deposit           10,000.00       10,000.00              -

   BANK ACCOUNT MANAGEMENT SYSTEM
 1. Open a new account
 2. View account details
 3. Deposit
 4. Withdraw
 5. Transfer funds
 6. View transaction history
 7. View summary statistics
 8. Exit
Deposit successful. New balance: Rs. 12,500.00

   BANK ACCOUNT MANAGEMENT SYSTEM
 1

In [ ]:
#Pandas Groupby analysis

In [24]:
def transactions_to_dataframe(accounts: Dict[str, Account]):
    """ Convert all transactions into a Dataframe"""
    import pandas as pd
    rows = []
    for acc_no, acc in accounts.items():
        for t in acc.transactions:
            rows.append({
                "account_number": acc_no,
                "holder_name": acc.holder_name,
                "account_type": acc.account_type,
                "timestamp": t.timestamp,
                "type": t.ttype,
                "amount": t.amount,
                "balance_after": t.balance_after,
                "counterparty": t.counterparty,
            })
    return pd.DataFrame(rows)


 
df = transactions_to_dataframe(reloaded)
print(df.to_string(index=False))
print("\nPer-account, per-type totals:")
print(df.groupby(["account_number", "type"])["amount"].sum())

account_number   holder_name account_type           timestamp         type   amount  balance_after counterparty
       ACC1001 Bhanwar Singh      savings 2026-10-05 19:40:31      deposit  50000.0        50000.0          NaN
       ACC1001 Bhanwar Singh      savings 2026-10-05 19:41:41      deposit   1500.0        51500.0          NaN
       ACC1001 Bhanwar Singh      savings 2026-10-05 19:41:41   withdrawal    800.0        50700.0          NaN
       ACC1001 Bhanwar Singh      savings 2026-10-05 19:42:29 transfer_out   1200.0        49500.0      ACC1002
       ACC1002 Kavita Kanwar      current 2026-10-05 19:40:31      deposit 200000.0       200000.0          NaN
       ACC1002 Kavita Kanwar      current 2026-10-05 19:41:41      deposit    750.0       200750.0          NaN
       ACC1002 Kavita Kanwar      current 2026-10-05 19:42:29  transfer_in   1200.0       201950.0      ACC1001
       ACC1003   Aarav Mehta      savings 2026-10-05 19:50:49      deposit  10000.0        10000.0      

In [33]:
accounts = load_accounts()
accounts = main_menu(accounts)


   BANK ACCOUNT MANAGEMENT SYSTEM
 1. Open a new account
 2. View account details
 3. Deposit
 4. Withdraw
 5. Transfer funds
 6. View transaction history
 7. View summary statistics
 8. Exit


Select an option (1-8):  7
Account number:  ACC1001



Summary Statistics -- ACC1001
------------------------------------------
 Total Deposits          : Rs. 51,500.00
 Total Withdrawals       : Rs. 15,350.00
 Average Txn Amount      : Rs. 11,141.67
 Largest Txn Amount      : Rs. 50,000.00
 Txn Amount Std. Dev.    : Rs. 17,820.08
 Total Transaction Count : 6
------------------------------------------

   BANK ACCOUNT MANAGEMENT SYSTEM
 1. Open a new account
 2. View account details
 3. Deposit
 4. Withdraw
 5. Transfer funds
 6. View transaction history
 7. View summary statistics
 8. Exit


Select an option (1-8):  8


Data saved. Goodbye!
